In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/ab_data.csv')
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())
print(df['group'].value_counts())
print(df['converted'].value_counts())

(294478, 5)
   user_id                   timestamp      group landing_page  converted
0   851104  2017-01-21 22:11:48.556739    control     old_page          0
1   804228  2017-01-12 08:01:45.159739    control     old_page          0
2   661590  2017-01-11 16:55:06.154213  treatment     new_page          0
3   853541  2017-01-08 18:28:03.143765  treatment     new_page          0
4   864975  2017-01-21 01:52:26.210827    control     old_page          1
user_id         int64
timestamp         str
group             str
landing_page      str
converted       int64
dtype: object
user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64
group
treatment    147276
control      147202
Name: count, dtype: int64
converted
0    259241
1     35237
Name: count, dtype: int64


In [2]:
# control should always see old_page
# treatment should always see new_page
mismatched = df[
    ((df['group'] == 'control') & (df['landing_page'] == 'new_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page'))
]
print(f"Mismatched rows: {mismatched.shape[0]}")
print(mismatched.head())

Mismatched rows: 3893
     user_id                   timestamp      group landing_page  converted
22    767017  2017-01-12 22:58:14.991443    control     new_page          0
240   733976  2017-01-11 15:11:16.407599    control     new_page          0
308   857184  2017-01-20 07:34:59.832626  treatment     old_page          0
327   686623  2017-01-09 14:26:40.734775  treatment     old_page          0
357   856078  2017-01-12 12:29:30.354835  treatment     old_page          0


In [3]:
print(f"Total rows: {df.shape[0]}")
print(f"Unique users: {df['user_id'].nunique()}")
print(f"Duplicate users: {df.shape[0] - df['user_id'].nunique()}")
print(df[df['user_id'].duplicated()].head())

Total rows: 294478
Unique users: 290584
Duplicate users: 3894
       user_id                   timestamp      group landing_page  converted
2656    698120  2017-01-15 17:13:42.602796    control     old_page          0
2893    773192  2017-01-14 02:55:59.590927  treatment     new_page          0
7500    899953  2017-01-07 03:06:54.068237    control     new_page          0
8036    790934  2017-01-19 08:32:20.329057  treatment     new_page          0
10218   633793  2017-01-17 00:16:00.746561  treatment     old_page          0


In [4]:
# drop mismatched rows
df = df[
    ((df['group'] == 'control') & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))
]
print(f"After removing mismatches: {df.shape}")

# drop duplicate user_ids, keep first occurrence
df = df.drop_duplicates(subset='user_id', keep='first')
print(f"After removing duplicates: {df.shape}")

After removing mismatches: (290585, 5)
After removing duplicates: (290584, 5)


In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(df['timestamp'].dtype)
print(f"Test start: {df['timestamp'].min()}")
print(f"Test end: {df['timestamp'].max()}")
print(f"Test duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")

datetime64[us]
Test start: 2017-01-02 13:42:05.378582
Test end: 2017-01-24 13:41:54.460509
Test duration: 21 days


In [6]:
print(df['group'].value_counts())
print(f"\nControl conversion rate: {df[df['group']=='control']['converted'].mean()*100:.4f}%")
print(f"Treatment conversion rate: {df[df['group']=='treatment']['converted'].mean()*100:.4f}%")
print(f"Absolute difference: {(df[df['group']=='treatment']['converted'].mean() - df[df['group']=='control']['converted'].mean())*100:.4f}%")
print(f"\nFinal shape: {df.shape}")

group
treatment    145310
control      145274
Name: count, dtype: int64

Control conversion rate: 12.0386%
Treatment conversion rate: 11.8808%
Absolute difference: -0.1578%

Final shape: (290584, 5)
